# Cross-encoder for intent-api (phase 06)
Scores each (question, option) pair with one logit; the answer is a softmax over the offered options.
Run on Colab with a GPU runtime. Upload `train.jsonl` and `heldout.jsonl` (invented questions only).
Change `MODEL` to `microsoft/deberta-v3-small` to try the bigger candidate.

In [ ]:
!pip install -q -U huggingface_hub transformers "optimum[onnxruntime]" accelerate
print("restart the session now (Runtime > Restart session), then run from the next cell")

In [ ]:
import json, math, random, shutil
import numpy as np
import torch
from google.colab import files
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL = "distilbert-base-uncased"
EPOCHS = 4
LEARNING_RATE = 3e-5
BATCH_ITEMS = 16
MAX_LENGTH = 96
SEED = 7

uploaded = files.upload()

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

def load(name):
    return [json.loads(line) for line in open(name) if line.strip()]

train_all = load("train_v2.jsonl")
heldout = load("heldout_v2.jsonl")
random.shuffle(train_all)
calibration_size = min(400, len(train_all) // 10)
calibration, train = train_all[:calibration_size], train_all[calibration_size:]
print(len(train), len(calibration), len(heldout))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).to(device)

def pairs(item):
    spec = item["questions"]["choice"]
    names = list(spec["criteria"])
    texts_b = [f"{spec['instructions']} {name}: {spec['criteria'][name]}" for name in names]
    return names, [item["state"]["question"]] * len(names), texts_b

def encode(texts_a, texts_b):
    return tokenizer(texts_a, texts_b, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(device)

def item_logits(model, batch):
    all_names, all_a, all_b, sizes = [], [], [], []
    for item in batch:
        names, texts_a, texts_b = pairs(item)
        all_names.append(names)
        all_a += texts_a
        all_b += texts_b
        sizes.append(len(names))
    logits = model(**encode(all_a, all_b)).logits.squeeze(-1)
    return all_names, torch.split(logits, sizes)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
steps = EPOCHS * math.ceil(len(train) / BATCH_ITEMS)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min(1.0, (s + 1) / (0.1 * steps)) * max(0.0, (steps - s) / steps))

for epoch in range(EPOCHS):
    model.train()
    random.shuffle(train)
    total = 0.0
    for start in range(0, len(train), BATCH_ITEMS):
        batch = train[start:start + BATCH_ITEMS]
        names, grouped = item_logits(model, batch)
        loss = sum(
            torch.nn.functional.cross_entropy(logits.unsqueeze(0), torch.tensor([option_names.index(item["expected"]["choice"])], device=device))
            for item, option_names, logits in zip(batch, names, grouped)
        ) / len(batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total += loss.item() * len(batch)
    print(epoch, round(total / len(train), 4))

In [ ]:
def collect(score_batch, items):
    out = []
    for start in range(0, len(items), 32):
        batch = items[start:start + 32]
        for item, logits in zip(batch, score_batch(batch)):
            names = list(item["questions"]["choice"]["criteria"])
            out.append((logits, names.index(item["expected"]["choice"])))
    return out

@torch.no_grad()
def torch_scores(batch):
    model.eval()
    return [l.cpu().numpy() for l in item_logits(model, batch)[1]]

def softmax(x, t):
    z = np.exp((x - x.max()) / t)
    return z / z.sum()

def metrics(records, t):
    results = []
    for logits, target in records:
        p = softmax(logits, t)
        results.append((p.max(), int(p.argmax()) == target))
    accuracy = np.mean([ok for _, ok in results])
    ece = 0.0
    for i in range(10):
        bucket = [(c, ok) for c, ok in results if i / 10 < c <= (i + 1) / 10]
        if bucket:
            ece += len(bucket) / len(results) * abs(np.mean([c for c, _ in bucket]) - np.mean([ok for _, ok in bucket]))
    return round(float(accuracy), 4), round(float(ece), 4)

calibration_records = collect(torch_scores, calibration)
temperature = min((t / 10 for t in range(5, 80)), key=lambda t: -np.mean([math.log(softmax(l, t)[y] + 1e-9) for l, y in calibration_records]))
print("temperature", temperature)
print("heldout accuracy, ece", metrics(collect(torch_scores, heldout), temperature))

model.save_pretrained("torch_model")
tokenizer.save_pretrained("torch_model")
shutil.make_archive("torch_model", "zip", ".", "torch_model")
files.download("torch_model.zip")

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ORTModelForSequenceClassification.from_pretrained("torch_model", export=True).save_pretrained("onnx_fp32")
tokenizer.save_pretrained("onnx_fp32")
for name, config in {"onnx_int8_avx2": AutoQuantizationConfig.avx2(is_static=False, per_channel=False), "onnx_int8_arm64": AutoQuantizationConfig.arm64(is_static=False, per_channel=False)}.items():
    ORTQuantizer.from_pretrained("onnx_fp32").quantize(save_dir=name, quantization_config=config)
    tokenizer.save_pretrained(name)

In [ ]:
def onnx_scores_for(directory):
    ort = ORTModelForSequenceClassification.from_pretrained(directory)

    def run(batch):
        all_a, all_b, sizes = [], [], []
        for item in batch:
            _, texts_a, texts_b = pairs(item)
            all_a += texts_a
            all_b += texts_b
            sizes.append(len(texts_a))
        inputs = tokenizer(all_a, all_b, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt")
        logits = ort(**inputs).logits.squeeze(-1).numpy()
        return np.split(logits, np.cumsum(sizes)[:-1])

    return run

for directory in ["onnx_fp32", "onnx_int8_avx2"]:
    print(directory, metrics(collect(onnx_scores_for(directory), heldout), temperature))

In [ ]:
json.dump({"model": MODEL, "temperature": temperature, "max_length": MAX_LENGTH}, open("onnx_int8_avx2/calibration.json", "w"))
json.dump({"model": MODEL, "temperature": temperature, "max_length": MAX_LENGTH}, open("onnx_int8_arm64/calibration.json", "w"))
shutil.make_archive("cross_encoder", "zip", ".", "onnx_int8_avx2")
shutil.make_archive("cross_encoder_arm64", "zip", ".", "onnx_int8_arm64")
files.download("cross_encoder.zip")
files.download("cross_encoder_arm64.zip")